<h1 style="font-size: 1.6rem; font-weight: bold">ITO5163: Introduction to Cybersecurity</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 4: Authentication</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Authentication**

Security systems rely on three closely related ideas that are easy to mix up:

| Concept            | Question It Answers          | Example                                  |
|--------------------|------------------------------|------------------------------------------|
| **Identification** | Who do you **claim** to be?  | Entering a username                      |
| **Authentication** | Can you **prove** it?        | Entering the correct password            |
| **Authorisation**  | What are you **allowed** to do? | Being allowed to open a particular file |

**Authentication** is how a system checks that a user **is who they claim to be**, based on the information they provide. It's part of **access control**, and **authorisation only happens after authentication succeeds**. A username alone proves nothing, since anyone can type someone else's, so authentication methods have become more sophisticated over time.

During authentication, the user provides **credentials** to be checked. The most common is a **username and password**, but PINs, tokens, biometrics, or a combination of these are also used.

#### **Two-Party Authentication**

```mermaid
sequenceDiagram
    participant S as Supplicant
    participant A as Authenticator
    S->>A: Identity and credential(s)
    A->>S: Successful or failed authentication
```

- **Two-Party Authentication**:
  - **Supplicant**: the user or device trying to prove its identity
  - **Authenticator**: checks the credentials and decides whether authentication succeeds
  - The authenticator is the trusted party, so if it's compromised, the whole process is compromised
  - The supplicant talks to the authenticator **directly**

#### **Three-Party Authentication**

```mermaid
sequenceDiagram
    participant S as Supplicant
    participant R as Responder
    participant A as Authenticator
    S->>R: 1. Identity and credential(s)
    R->>A: 2. Identity and credential(s)
    A->>R: 3. Successful or failed authentication
    R->>S: 4. Successful or failed authentication
```

- **Three-Party Authentication**:
  - Adds a **responder**: an intermediary the supplicant contacts first
  - The responder **can't check the credentials itself**, but the authenticator trusts it to pass them on
  - The supplicant **never talks to the authenticator directly**, which keeps the authenticator protected
  - Credentials are checked in **one central place**, so many responders can share one authenticator
  - Example: logging in to university Wi-Fi. Your laptop (supplicant) talks to the Wi-Fi access point (responder), which forwards your login to a central server (authenticator)

#### **Authentication Factors**

Authentication usually follows a **challenge-response** model: the system presents a challenge, and the user must give a valid response. A valid response leads to success. An invalid one leads to failure or further challenges. The response can be based on several different **factors**.

| Factor               | Based On                          | Examples                                |
|----------------------|-----------------------------------|-----------------------------------------|
| Something you know   | A secret                          | Password, PIN                           |
| Something you have   | A physical object                 | Bank token, authenticator app, smart card |
| Something you are    | A physical trait (biometrics)     | Fingerprint, face, iris                 |
| Something you process | Performing a task with a secret  | Solving a challenge using a secret pattern |
| Somewhere you are    | Location                          | GPS, IP address                         |
| Somebody you know    | Social relationships              | A friend vouching for you               |

#### **Something You Know**

```mermaid
flowchart LR
    P1["Registration:<br>password"] --> T["Password template<br>(salted hash)"] --> DB[("Password database")] --> CMP{"Compare"}
    P2["Login:<br>password"] --> H["Same salted hash"] --> CMP
    CMP --> R["Match / No match"]
```

- **Something You Know**: a secret only the user should know, such as a **password** or **PIN**
  - **Registration**: the user chooses a password (and sometimes a username), and the authenticator stores it
  - **Login**: the user enters it again, and it's compared with what's stored
  - **Match**: access granted. **No match**: access denied
  - Only the **password or PIN** is the authentication factor. The username is just **identification**
  - Passwords should be stored as a **salted hash**, never as plain text, so a stolen database doesn't reveal them

#### **Something You Have**

The user proves they have a physical device, such as a bank's **authentication token** or an **authenticator app** on their phone. The device shows a **one-time password (OTP)** that the user types in. The device and server share a **secret key $K$**, which is what makes the code impossible to predict for anyone without the device.

**Synchronous (time-based) tokens**: the device and server keep their clocks in sync. Both compute the code from the current time $t$.

```mermaid
sequenceDiagram
    participant T as Authentication token
    participant A as Authentication server
    Note over T,A: Both share a secret key K and keep their clocks in sync
    T->>A: At time t: PWDt = f(K, t)
    Note over A: Computes f(K, t) itself<br>Received PWDt == computed PWDt ?
    A->>T: Successful or failed authentication
```

**Asynchronous (challenge-response) tokens**: no clock is needed. The server sends a challenge $x$, the user enters it into the token, and the token computes the response.

```mermaid
sequenceDiagram
    participant T as Authentication token
    participant A as Authentication server
    Note over T,A: Both share a secret key K
    A->>T: 1. Numerical challenge x
    T->>A: 2. PWDx = f(K, x)
    Note over A: 3. Computes f(K, x) itself<br>Received PWDx == computed PWDx ?
    A->>T: 4. Successful or failed authentication
```

| Type | Input to $f$ | Standard | Needs Clock Sync? |
|------|--------------|----------|-------------------|
| Time-based OTP (**TOTP**) | Secret key + current time | RFC 6238 | Yes |
| Counter-based OTP (**HOTP**) | Secret key + a counter that goes up each use | RFC 4226 | No, but counters must stay in step |
| Challenge-response | Secret key + server's challenge | e.g. OCRA, RFC 6287 | No |

- **Something You Have**:
  - The password is **different every time**, so a stolen code is useless shortly after
  - Without the secret key $K$, no one can compute the right code
  - Weakness: the device can be lost or stolen

#### **Something You Are**

```mermaid
flowchart LR
    C1["Registration:<br>capture"] --> E1["Extraction"] --> X1["Transformation"] --> BT["Biometric template"] --> DB[("Template database")] --> CMP{"Close enough?<br>(score ≥ threshold)"}
    C2["Login:<br>capture"] --> E2["Extraction"] --> X2["Transformation"] --> BI["Biometric information"] --> CMP
    CMP --> R["Match / No match"]
```

- **Something You Are**: uses **biometrics**, physical traits of the user
  - Examples: fingerprints, palm prints, hand or face geometry, iris and retina scans
  - **Registration**: capture the trait, extract its key features, and store them as a **template**
  - **Login**: capture the trait again and compare it with the stored template
  - Works like "something you know", but the match is **approximate**: two scans are never identical, so the system checks whether they're **close enough**

- **Errors**:
  - **False Acceptance Rate (FAR)**: an impostor is accepted
  - **False Rejection Rate (FRR)**: the real user is rejected
  - A stricter threshold lowers FAR but raises FRR

- **Weaknesses**:
  - Traits can **change over time**, through ageing or injury
  - Biometrics **can't be changed** if they're stolen, unlike a password
  - Some traits can be faked, such as a photo of a face

#### **Something You Process**

- **Something You Process**: the user completes a **task** using a secret shared with the authenticator
  - The result of the task becomes a code used to log in
  - To stop attacks, the challenge can change, or its values can be shuffled so they're never in the same place
  - Not commonly used
  - Disadvantage: users may not always be able to perform the task, and it can depend on their device and location

#### **Somewhere You Are**

```mermaid
flowchart LR
    A["User logs in to a banking<br>system on a computer"] --> B["Find the geolocation of<br>the user's mobile device"] --> C["Compare it with the IP<br>location of the computer"] --> D{"Are the locations<br>close to each other?"}
    D -- "Yes" --> E["Access granted"]
    D -- "No" --> F["End the session"]
```

- **Somewhere You Are** (location-based authentication):
  - Uses the user's **location** as evidence of identity
  - **IP address location** isn't always accurate, and can be hidden with a VPN
  - **GPS** on mobile devices is more accurate, so this works better on phones
  - Usually used as an **extra check** rather than on its own
  - **Banks** use it to detect fraud, e.g. flagging a payment from a different country to the user's last login

#### **Somebody You Know**

```mermaid
sequenceDiagram
    participant A as Alice (asker)
    participant B as Bob (helper)
    participant S as Server
    A->>B: 1. "Bob, I need help. I haven't got my token!"
    Note over B: 2. Bob checks it's really Alice
    B->>S: 3. Bob logs in with his own credentials
    S->>B: 4. "The vouchcode for Alice is axbycz9291"
    B->>A: 5. "Alice, your vouchcode is axbycz9291"
    A->>S: 6. Password + vouchcode
    S->>A: 7. "Alice, this is your temporary password"
```

- **Somebody You Know** (social authentication):
  - Based on **social relationships**: someone the user knows **vouches** for them
  - The least-used method, intended as a **last resort**, e.g. when a user has lost their token
  - Users register in advance as an **asker** and/or **helper**, linked to another user
  - The helper logs in and gets a **vouchcode** for the asker, who uses it to log in
  - Security depends on the **helper** correctly checking the asker's identity outside the system, so it's open to **social engineering**

#### **Multi-Factor Authentication (MFA)**

Combining factors makes authentication much stronger, because an attacker would need to steal **different kinds** of evidence.

- **Multi-Factor Authentication**:
  - Uses factors from **different categories**, e.g. a password (know) + a phone code (have)
  - Two passwords is still **single-factor**, since both are "something you know"
  - **Two-factor authentication (2FA)** is the most common form

<br>

***

### **Password-Based Authentication**

Passwords have been used in computing for a long time. They're simple, which is why they're still the **main authentication method** in most systems. To use them well, we need to understand their **strengths and weaknesses**, and the different ways they can be stored.

#### **Password Strength**

A good password should be **easy for the user to remember** but **hard for anyone else to guess**. In practice, these two goals often conflict.

The number of possible passwords is $c^L$, where $c$ is the number of possible characters and $L$ is the length. For an 8-character password:

| Character Set             | Characters $c$ | Possible Passwords $c^8$ | Approx. |
|---------------------------|----------------|--------------------------|---------|
| Digits only               | 10             | 100,000,000              | $2^{26.6}$ |
| Lowercase letters         | 26             | 208,827,064,576          | $2^{37.6}$ |
| Upper + lowercase         | 52             | $5.3 \times 10^{13}$     | $2^{45.6}$ |
| Letters + digits          | 62             | $2.2 \times 10^{14}$     | $2^{47.6}$ |
| All printable keyboard characters | 95     | $6.6 \times 10^{15}$     | $2^{52.6}$ |
| Any 8-bit value (theoretical) | 256        | $1.8 \times 10^{19}$     | $2^{64}$ |

In theory, if every character could be any of 256 values, there would be $256^8 = 2^{64}$ possibilities. In reality, users can only type about 95 characters, and they **don't choose randomly**. They use common words, names, birthdays and patterns, which attackers try first, so the real number of guesses needed is **far smaller**.

#### **Password Policies**

To push users towards stronger passwords, administrators often enforce rules such as:

- At least one **number**
- At least one **uppercase** and one **lowercase** letter
- At least one **special character**
- A **minimum length**, and sometimes a maximum
- **Can't reuse** a previous password
- **Can't contain** a name or proper noun

Current guidance (NIST SP 800-63B) puts **length first**: long passphrases are both easier to remember and harder to guess. It also recommends checking new passwords against lists of **known breached passwords**, rather than relying only on composition rules.

#### **Storing Passwords**

Even a strong password is useless if it's **stored insecurely**. There are three main approaches.

| Method         | Stored Value                   | If the Database Is Stolen                    | Recommended? |
|----------------|--------------------------------|----------------------------------------------|--------------|
| **Plaintext**  | The password itself            | Every password is exposed immediately        | Never        |
| **Encrypted**  | $E(K, \text{password})$        | Every password is exposed if the key is stolen too | No     |
| **Hashed**     | $H(\text{password})$           | Attacker must guess each password            | Only with a salt |
| **Salted + slow hash** | salt, $H(\text{salt} \parallel \text{password})$ | Attacker must guess each password separately, slowly | **Yes** |

#### **Plaintext Storage**

The **least secure** option. Anyone with access to the system or database can **read, copy or change** every password.

#### **Encrypted Storage**

The password is encrypted with a **symmetric** or **public-key** cipher, and the ciphertext is stored instead.

- **Encrypted Storage**:
  - More secure than plaintext, but encryption is **reversible**: anyone with the key can recover every password
  - Raises design questions: symmetric or public-key? One key for all passwords, or one per user?
  - Creates a **key management problem** for systems with thousands of users
  - The system never actually needs to recover the password, only to **check** it, so encryption isn't needed

#### **Hashed Storage**

A **hash function** takes an input of any length and always produces a **fixed-size output**. It's **one-way**: easy to compute, but infeasible to reverse. The system stores only the hash, and at login it hashes the entered password and compares the two.

| Password   | MD5                                | SHA-1                                      |
|------------|------------------------------------|--------------------------------------------|
| admin      | `21232f297a57a5a743894a0e4a801fc3` | `d033e22ae348aeb5660fc2140aec35850c4da997` |
| password   | `5f4dcc3b5aa765d61d8327deb882cf99` | `5baa61e4c9b93f3f0682250b6cf8331b7ee68fd8` |
| superman   | `84d961568a65073a3bcf0eb216b2a576` | `18c28604dd31094a8d69dae60f1bcd347f1afc5a` |
| D0raem0n   | `3cd70ea8040d94980c4644d6a3e29b3f` | `058ded9bbbadf334515454d8030bbff8c50b0f89` |
| MtFbwY     | `b4e7d9a5aed04ac29736a311433dfde6` | `73160bfef9d2b95f4fce443d4283a13105279bb8` |

- **Hashed Storage**:
  - The system never stores the real password
  - Every hash is the same length, so it doesn't reveal the password's length
  - **Same password always gives the same hash**, so two users with the same password have the same hash

#### **Precomputed Table Attacks**

Because the same password always gives the same hash, an attacker who knows the hash function can **compute the hashes of millions of common passwords in advance**. Then they just look up each stolen hash. **Rainbow tables** are a space-saving form of these precomputed tables.

For example, anyone who sees `5f4dcc3b5aa765d61d8327deb882cf99` in a stolen MD5 database instantly knows the password is `password`. Even `D0raem0n` (with zeros instead of o's) is a common substitution, so it's likely in these tables.

#### **Salting**

A **salt** is a **random value** added to the password before hashing. Each user gets their **own unique salt**, which is stored **alongside the hash**. The salt doesn't need to be secret.

```mermaid
flowchart LR
    P1["Registration:<br>password"] --> S1["+ new random salt"] --> H1["Slow hash<br>(e.g. bcrypt, Argon2)"] --> DB[("Store: username,<br>salt, hash")]
    P2["Login:<br>password"] --> S2["+ salt from database"] --> H2["Same slow hash"] --> CMP{"Hashes<br>match?"}
    DB --> CMP
    CMP -- "Yes" --> OK["Access granted"]
    CMP -- "No" --> NO["Access denied"]
```

| User  | Password   | Salt   | Stored Hash (SHA-256, first 16 hex digits) |
|-------|------------|--------|--------------------------------------------|
| -     | password   | none   | `5e884898da280471` (in every precomputed table) |
| Alice | password   | `x7Qp` | `c51926f5dc0d5dc8`                         |
| Bob   | password   | `R2mz` | `44af260fb1c2fb3b`                         |

- **What Salting Does**:
  - **Defeats precomputed tables**: the attacker would need a separate table for every possible salt
  - **Hides identical passwords**: Alice and Bob have the same password but different hashes
  - Forces the attacker to crack **each password separately**
  - Salts should be **long** (e.g. 16 bytes) and **random**, and never reused

- **What Salting Doesn't Do**:
  - It doesn't make a **weak password strong**. An attacker can still try common passwords against one user's salt
  - That's why password hashes should also be **deliberately slow**

#### **Slow Hash Functions**

MD5 and SHA-1 are designed to be **fast**, which helps attackers: a single GPU can try **billions** of guesses per second. MD5 and SHA-1 also have known weaknesses. Password storage should use **slow, purpose-built** functions, which make each guess expensive.

- **Recommended**: **Argon2**, **bcrypt**, **scrypt**, **PBKDF2**
- They include a salt automatically, and have a **work factor** that can be increased as computers get faster

#### **Defence in Depth**

Password storage techniques mostly **delay or deter** attacks rather than make them impossible. Usernames and passwords won't be replaced any time soon, so they're combined with other protections:

- **Lockouts and timeouts**: block or slow down an account after several failed attempts
- **Rate limiting**: limit how many login attempts can be made per minute
- **Multi-factor authentication**: a stolen password alone isn't enough to log in

<br>

***

### **Biometric Authentication**

Passwords are the most common way to authenticate, but they have limits: they can be forgotten, guessed or stolen. Many systems are now moving to **Biometrics**: measurable **physiological and behavioural characteristics** of a person. The most common are the **face**, **fingerprint** and **iris**.

Biometrics are hard to use for online logins, but they're now standard on **laptops and smartphones**. For example, Apple introduced **Face ID** in 2017, and most modern phones unlock with a face or fingerprint.

#### **Why Biometrics?**

Biometrics are based on natural characteristics that are **hard to duplicate**. **Sir Francis Galton**, who pioneered the study of fingerprints, estimated the chance of two people having the same fingerprint as about **1 in 64 billion**. With about **8 billion** people on Earth, that makes fingerprints very effective for telling people apart.

Biometrics aren't impossible to fake, though. Fingerprints can be copied onto moulds, and faces can be shown in photos. Modern systems use **liveness detection** to check that a real, living person is present.

#### **Types of Biometrics**

```mermaid
flowchart TB
    B["Biometrics"] --> P["Physiological<br>(what you are)"]
    B --> BH["Behavioural<br>(what you do)"]
    P --> P1["Fingerprint"]
    P --> P2["Face"]
    P --> P3["Iris / retina"]
    P --> P4["Palm print /<br>hand geometry"]
    BH --> B1["Voice"]
    BH --> B2["Handwritten<br>signature"]
    BH --> B3["Typing pattern<br>(keystroke dynamics)"]
    BH --> B4["Walking style<br>(gait)"]
```

- **Physiological Biometrics**: physical features of the body
  - Examples: fingerprint, face, palm print, hand geometry, retina, iris
  - Usually **easy to capture** (retina scans are an exception, as they need the eye close to a scanner)
  - Generally **very accurate** when captured correctly
  - Can **change over time** due to age, illness or injury

- **Behavioural Biometrics**: patterns in how a person acts
  - Examples: voice, handwritten signature, typing rhythm, walking style
  - Can often be collected **passively** and **continuously**, e.g. checking typing rhythm throughout a session
  - Usually **less accurate**, because behaviour varies with mood, tiredness or illness
  - Can be imitated: signatures can be forged, and voices can be recorded or faked

| Feature          | Physiological                 | Behavioural                         |
|------------------|-------------------------------|-------------------------------------|
| Based on         | Body features                 | Actions and habits                  |
| Accuracy         | Generally high                | Generally lower                     |
| Day-to-day variation | Low                       | Higher (mood, tiredness, illness)   |
| Long-term change | Age, illness, injury          | Habits can drift over time          |
| Collection       | Usually needs a deliberate scan | Often passive and continuous      |

#### **Properties of an Ideal Biometric**

- **Universality**: every user has it
- **Uniqueness**: it's different for every person, so people can be told apart
- **Permanence**: it stays the same for a long time, so it doesn't need re-registering often
- **Collectability**: it's easy to capture and measure
- **Acceptability**: people are willing to have it collected
- **Performance**: it can be matched quickly and accurately
- **Circumvention**: it's hard to fake or fool

No biometric is perfect on all of these. For example, fingerprints score well on uniqueness and permanence, but some people have worn or missing fingerprints, which fails universality.

#### **The Biometric Authentication Process**

The process is similar to password authentication:

- **Registration (enrolment)**: the biometric is captured, its key features are extracted, and the result is stored as a **template**
- **Authentication**: the biometric is captured again, processed the same way, and **compared** with the stored template
- **Decision**: if they're close enough, access is granted

Biometric systems can work in two modes:

| Mode | Question | Compares Against | Example |
|------|----------|------------------|---------|
| **Verification (1:1)** | Are you who you claim to be? | One stored template | Unlocking your own phone |
| **Identification (1:N)** | Who are you? | Every template in the database | Searching police fingerprint records |

#### **Feature Extraction**

Biometric systems don't store a photo of the trait. They extract **detailed features**, such as the ridge endings and splits in a fingerprint (called **minutiae**), and compare those. The algorithms are beyond the scope of this unit.

```mermaid
flowchart LR
    C1["Registration:<br>biometric data capture"] --> P1["Biometric data<br>processing"] --> DB[("Storing<br>biometric data")] --> M{"Do they<br>match?"}
    C2["Login:<br>biometric data capture"] --> P2["Biometric data<br>processing"] --> M
    M -- "Yes" --> Y["Match"]
    M -- "No" --> N["No match"]
```

#### **Matching Errors**

Unlike passwords, two scans of the same finger are **never exactly identical**, because of angle, pressure, lighting or dirt. The system calculates a **similarity score** and accepts the user if it's above a **threshold**. More detail doesn't always mean better results, and the threshold has to be chosen carefully.

| Error | Also Called | What Happens | Risk |
|-------|-------------|--------------|------|
| **False Acceptance Rate (FAR)** | False positive | An **impostor is accepted** | Security breach |
| **False Rejection Rate (FRR)** | False negative | The **real user is rejected** | Frustration, poor usability |

| Threshold | FAR | FRR | Suits |
|-----------|-----|-----|-------|
| **Strict** (high) | Low | High | High-security systems, e.g. a bank vault |
| **Lenient** (low) | High | Low | Convenience, e.g. unlocking a phone |
| **Equal Error Rate (EER)** | FAR = FRR | FAR = FRR | Comparing different systems: lower EER means more accurate |

#### **Advantages and Disadvantages**

- **Advantages**:
  - Nothing to remember or carry
  - Hard to share, lose or forget
  - Fast and convenient

- **Disadvantages**:
  - **Can't be changed if stolen**: you can reset a password, but not a fingerprint
  - **Privacy concerns**: biometric data is sensitive personal information
  - **Matching errors**: some impostors may be accepted, and some real users rejected
  - **Spoofing**: copies such as moulds or photos can sometimes fool scanners
  - Usually best used as **one factor** in multi-factor authentication

<br>

***

### **Multi-Factor Authentication**

For many systems today, **one authentication method isn't enough**. Even if the person logging in is usually the real user, organisations add extra protection in case a password is stolen. This is done with **Two-Factor Authentication (2FA)** and **Multi-Factor Authentication (MFA)**. A common example is online banking: you log in with a username and password, then confirm a transaction with a **code sent by SMS**.

#### **Why Passwords Alone Aren't Enough**

- **Weak passwords**: complex passwords are hard to remember, especially for accounts used rarely, so people choose simple ones
- **Too many accounts**: best practice is a different password for each account, but that's hard to remember
- **Password reuse**: many people use the **same password everywhere**
  - If one site is breached, attackers try the same username and password on other sites
  - This is called **credential stuffing**
  - A second factor stops it, because the stolen password alone isn't enough

#### **Two-Factor Authentication (2FA)**

```mermaid
sequenceDiagram
    participant U as User
    participant S as Server
    participant P as User's phone
    U->>S: 1. Username + password (something you know)
    Note over S: Checks the password
    S->>P: 2. Sends a one-time secret code, e.g. 587764
    P-->>U: Code shown on the phone (something you have)
    U->>S: 3. Enters the code as the second factor
    Note over S: Checks the code
    S->>U: 4. Access granted
```

- **Two-Factor Authentication**:
  - **First factor**: usually the username and password
  - **Second factor**: a different method, such as a code sent to a phone
  - Authentication and verification happen **twice**, once for each factor
  - The two factors must come from **different categories**: know, have or are
    - Password + SMS code (know + have) is 2FA ✓
    - Password + security question (know + know) is **not** 2FA ✗
  - **"Remember this device"**: many systems skip the second factor on a trusted device after the first successful login
    - The device stores a token, not the password, which reduces inconvenience for regular users
    - If that device is stolen, the protection is weaker

#### **Common Second Factors**

| Second Factor | Category | Strengths | Weaknesses |
|---------------|----------|-----------|------------|
| **SMS code** | Have (phone) | Easy, works on any phone | Can be intercepted or stolen through **SIM swapping**, and can be phished |
| **Authenticator app (TOTP)** | Have (phone) | Works offline, not tied to the phone number | Can still be phished, and is lost if the phone is lost without a backup |
| **Push notification** | Have (phone) | One tap to approve | **MFA fatigue**: attackers send repeated prompts until the user taps "approve" |
| **Hardware security key (FIDO2 / passkey)** | Have | **Resistant to phishing**, as it checks which website it's talking to | Costs money, and can be lost |
| **Biometric** | Are | Convenient, nothing to remember | Usually unlocks a device rather than being checked by the server |

#### **Multi-Factor Authentication (MFA)**

**Multi-Factor Authentication** extends 2FA to **two or more** factors. It's less convenient, but it's the standard for systems that need **strong security**. Alongside the usual know / have / are factors, some newer factors are being explored:

- **Time Factor**: login is only allowed **during set times**
  - e.g. an employee can only log in during working hours
  - The **server's** clock decides whether the request is inside the allowed window

- **Location Factor**: login is only allowed from the **region** the user is registered in
  - Attempts from outside that region are rejected
  - **IP addresses** alone aren't reliable, as they can be hidden with a VPN, so **GPS** is sometimes also used

- **Ambient Sound Factor**: checks that the user's **phone is near the computer** they're logging in from
  - Both the phone and the computer record the **background sound** for a few seconds at the same moment
  - If the two recordings are similar, the devices are in the same place
  - The user doesn't have to do anything, as long as the phone is nearby

| Factor | Checks | Weakness |
|--------|--------|----------|
| Time | **When** the user logs in | Doesn't stop an attacker who logs in during allowed hours |
| Location | **Where** the user logs in from | IP can be hidden with a VPN, and GPS can be faked |
| Ambient sound | Whether the phone is **near** the computer | Can fail in very quiet or very noisy places |

#### **Security vs Usability**

Authentication has to be **user-friendly** enough that people are willing to do it every time they log in.

- **Usability Considerations**:
  - Organisations can **require** MFA for their own staff
  - For public services, too much friction can drive users to **less secure alternatives** that are more convenient
  - Methods like **"remember this device"**, push notifications and passkeys make strong authentication easier to use

<br>

***

### **Methods and Threats**

Almost every system, whether used in person or over a network, needs **valid authentication** before it can be used. That makes authentication the **first thing attackers target**. In the **OWASP Top 10** list of web application risks, broken authentication was ranked **#2** in 2017. It's **#7**, as "Authentication Failures", in the current 2025 edition.

#### **Default Passwords**

Most devices come with a **default username and password** set by the manufacturer, such as `admin` / `admin`. These are meant to be **changed** by the user or administrator, but often aren't.

- **Default Passwords**:
  - Usually **very simple**, and listed in the product's **public documentation**
  - Often left unchanged on networking devices for easy access
  - Anyone who looks up the manual can log in, then reach the rest of the network
  - Example: the **Mirai botnet** (2016) took over hundreds of thousands of cameras and routers using default passwords
  - **Defence**: change default passwords on setup. Some countries, such as the UK, now ban devices that share one default password

#### **Eavesdropping (Sniffing)**

**Eavesdropping** means listening in on someone else's communication. On computer networks it's called **sniffing**.

- **Eavesdropping**:
  - **Wired networks** are harder to sniff unless a device on the network has been compromised
  - **Wireless networks** send data over radio signals, which anyone nearby can pick up
  - **Plaintext protocols** (Telnet, FTP, HTTP) send passwords unencrypted, so they can be read directly
  - **Defence**: never send passwords in plaintext. Use **encrypted protocols** such as HTTPS and SSH, so a sniffed password can't be read

#### **Replay Attacks**

A **Replay Attack** happens **after** successful eavesdropping. The attacker captures an authentication packet and **sends it again later**.

```mermaid
sequenceDiagram
    participant U as User
    participant X as Attacker
    participant S as Server
    U->>S: Encrypted login packet
    Note over X: Sniffs and saves a copy
    S->>U: Access granted
    Note over X: Later...
    X->>S: Sends the same encrypted packet again
    S->>X: Access granted (no check of when it was sent)
```

- **Replay Attacks**:
  - The attacker **doesn't need to decrypt** the password. They just resend the encrypted packet
  - It works if the server doesn't check **when** the packet was sent, or whether it's **new**
  - **Encryption alone doesn't stop it**

- **Defences**:
  - **Timestamps**: the server only accepts a packet within a short time window, which needs the clocks to be in sync
  - **Nonces**: the server sends a random, one-time challenge that must be included in the response, so no clock sync is needed
  - **Sequence numbers** and **one-time passwords**: each value can only be used once

#### **Man-in-the-Middle (MITM) Attacks**

In a **Man-in-the-Middle** attack, the attacker **pretends to be the real system**. The user connects to the attacker without realising it, and the attacker can read everything sent.

```mermaid
sequenceDiagram
    participant U as User
    participant X as Attacker
    participant S as Real server
    U->>X: Login details (thinks it's the real server)
    Note over X: Reads everything
    X->>S: Forwards the login details
    S->>X: Response
    X->>U: Forwards the response
    Note over U: Nothing looks wrong
```

- **Man-in-the-Middle**:
  - The attacker can **drop** the connection after stealing the data, which may make the user suspicious
  - Or they can **relay** everything between the user and the real server, so the user notices nothing
  - Common on **fake Wi-Fi hotspots**

- **Defences**:
  - Use **TLS (HTTPS)** and check the server's **certificate**, so the attacker can't pretend to be the real server
  - Be wary of **certificate warnings** or anything unusual about a familiar site
  - **Mutual authentication**: both sides prove their identity
  - A **VPN** protects traffic on untrusted networks like public Wi-Fi

#### **Password Guessing**

The most basic attack is simply **guessing the password**. Complex passwords are very hard to guess, but simple ones can be guessed quickly.

| Attack | How It Works | Defeated By |
|--------|--------------|-------------|
| **Brute force** | Tries **every possible combination** of characters | Long, complex passwords |
| **Dictionary** | Tries **dictionary words**, since people choose words they can remember | Avoiding real words |
| **Password dictionary** | Tries a list of **common and leaked passwords** collected from past breaches | Checking new passwords against breach lists |
| **Password spraying** | Tries a **few common passwords** against **many accounts** | MFA, and spotting failed logins across accounts |

- **Password Spraying**: trying only a few guesses per account **avoids lockouts**, which normally trigger after several failures on one account
- **Online vs Offline Guessing**:
  - **Online**: guessing through the login page, which is slowed down by lockouts and rate limits
  - **Offline**: guessing against a **stolen database of hashes**, with no limits, which is why slow, salted hashing matters

#### **Credential Stuffing**

**Credential Stuffing** takes advantage of people **reusing the same password** on many sites. An attacker gets a username and password from one breached site, then tries them automatically on many others.

- **Defences**:
  - Use a **different password** for every site, which is easiest with a **password manager**
  - **Multi-factor authentication**, so a stolen password alone isn't enough
  - Sites can check passwords against **lists of known breached passwords**

#### **Social Engineering**

**Social Engineering** tricks **people** instead of computers. The attacker gains the victim's trust, often by **pretending to be someone they trust**, to get information or access. It's one of the few attacks that doesn't need a computer at all.

- **In-Person Social Engineering**:
  - The attacker gets information face to face by building trust
  - Techniques mentioned in the course: neuro-linguistic programming, six degrees of separation and bar hopping
  - Also includes **tailgating**: following someone through a secure door

- **Phone Social Engineering**:
  - **Anger approach**: pretends to be a frustrated senior employee demanding urgent access
  - **Fear approach**: pretends to be tech support, says the account has been compromised, and asks for the current password to "reset" it
  - **Vishing** (voice phishing): phone calls, often with automated messages, that trick the victim into revealing information

- **Digital Social Engineering**:
  - **Phishing**: mass emails with links to **fake websites** that collect passwords, steal cookies or install malware
  - **Spear-phishing**: phishing aimed at a **specific person or group**, often using personal details to look convincing
  - **Whaling**: spear-phishing aimed at **senior executives**
  - **Smishing**: phishing by **SMS**
  - **Pretexting**: inventing a **believable story** (a pretext) to get the victim to share information or open a malicious attachment

| Type | Targets | Channel |
|------|---------|---------|
| Phishing | Everyone | Email, fake websites |
| Spear-phishing | A specific person or group | Email |
| Whaling | Senior executives | Email |
| Vishing | Anyone | Phone call |
| Smishing | Anyone | SMS |

#### **Summary of Threats and Defences**

| Threat | What the Attacker Does | Main Defence |
|--------|------------------------|--------------|
| Default passwords | Logs in with the manufacturer's password | Change it on setup |
| Eavesdropping | Captures passwords sent over the network | Encrypted protocols (HTTPS, SSH) |
| Replay attack | Resends captured login packets | Timestamps, nonces |
| Man-in-the-middle | Sits between the user and the server | TLS with certificate checks |
| Password guessing | Tries likely passwords | Strong passwords, lockouts, rate limits |
| Password spraying | Tries a few passwords on many accounts | MFA, monitoring failed logins |
| Credential stuffing | Reuses passwords stolen elsewhere | Unique passwords, MFA |
| Social engineering | Tricks people into giving information | User awareness and training |

<br>

***